In [1]:
# ==============================================================================
# 🛡️ HÜCRE 1: ÇEKİRDEK ORTAM, ZIRHLAMA VE BAĞIMLILIKLAR
# ==============================================================================
import os, sys, gc, subprocess, glob
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("📦 1. Kütüphaneler kuruluyor...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "opencv-python", "matplotlib", "scikit-image", "einops", "kornia",
                "timm", "yacs", "joblib", "natsort", "h5py", "tqdm", "ptflops",
                "seaborn", "addict", "future", "lmdb", "numpy", "pyyaml", "requests",
                "scipy", "yapf", "lpips", "cython", "cython_bbox", "pandas",
                "xmltodict", "loguru", "gdown", "lapx", "motmetrics", "filterpy",
                "thop", "faiss-cpu", "tabulate"])

print("📥 2. Repolar klonlanıyor (DeepRFT, LightStab, HybridSORT)...")
for repo, url in [("DeepRFT", "https://github.com/INVOKERer/DeepRFT.git -b AAAI2023"),
                  ("LightStab", "https://github.com/liutao23/LightStab.git"),
                  ("HybridSORT", "https://github.com/ymzis69/HybridSORT.git")]:
    if not os.path.exists(f'/content/{repo}'):
        os.system(f"git clone {url} /content/{repo}")

print("🛠️ 3. Sistem yamaları (NumPy 2.x & Headless Matplotlib) uygulanıyor...")
# LightStab Headless Yama
ls_file = "/content/LightStab/model/LightMotionEsitimation.py"
if os.path.exists(ls_file):
    with open(ls_file, "r") as f: c = f.read()
    with open(ls_file, "w") as f: f.write(c.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')"))

# HybridSORT NumPy 2.x Yaması
for py_file in glob.glob("/content/HybridSORT/**/*.py", recursive=True):
    try:
        with open(py_file, 'r', encoding='utf-8') as f: code = f.read()
        for old, new in [("np.float(", "float("), ("np.int(", "int("), ("np.bool(", "bool("),
                         ("astype(float32)", "astype(np.float32)"), ("dtype=float32", "dtype=np.float32"),
                         ("astype(int32)", "astype(np.int32)")]:
            code = code.replace(old, new)
        with open(py_file, 'w', encoding='utf-8') as f: f.write(code)
    except: pass

os.chdir("/content/HybridSORT")
if not os.path.exists("yolox.egg-info"):
    os.system("pip install -e . --no-build-isolation --no-deps -q")
os.chdir("/content")

print("✅ Hücre 1 Tamamlandı: Ortam %100 Hazır ve Zırhlı.")

Mounted at /content/drive
📦 1. Kütüphaneler kuruluyor...
📥 2. Repolar klonlanıyor (DeepRFT, LightStab, HybridSORT)...
🛠️ 3. Sistem yamaları (NumPy 2.x & Headless Matplotlib) uygulanıyor...
✅ Hücre 1 Tamamlandı: Ortam %100 Hazır ve Zırhlı.


In [25]:
# ==============================================================================
# 📂 HÜCRE 2: VERİSETİ BAĞLANTISI (MOT17-04) VE İLKLEME
# ==============================================================================
import os
import glob
import cv2
import torch
import shutil

# 1. Veriseti Yolları (Drive'a yeni yüklediğin MOT17-04 dizini)
dataset_base = "/content/drive/MyDrive/Spikedge_Staj/Tracking/MOT17_Dataset/train/MOT17-04-FRCNN"
img_dir = os.path.join(dataset_base, "img1")
gt_path = os.path.join(dataset_base, "gt/gt.txt")

image_files = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))

print("="*50)
print(f"📁 Seçilen Senaryo: MOT17-04-FRCNN")
print(f"🎞️ Toplam Kare Sayısı: {len(image_files)}")
print(f"📊 Ground Truth Konumu: {gt_path}")
print("="*50)

if not image_files:
    raise FileNotFoundError("❌ Görüntüler bulunamadı! Drive yolunu kontrol edin.")
if not os.path.exists(gt_path):
    raise FileNotFoundError("❌ gt.txt bulunamadı! Verisetinin eksiksiz yüklendiğinden emin olun.")

# 2. YOLOX Ağırlık Yükleme ve Zırhlı Taşıma
drive_yolox = "/content/drive/MyDrive/Spikedge_Staj/Tracking/pretrained/ocsort_x_mot17.pth.tar"
local_yolox = "/content/HybridSORT/pretrained/ocsort_x_mot17.pth.tar"

os.makedirs(os.path.dirname(local_yolox), exist_ok=True)
if os.path.exists(drive_yolox) and not os.path.exists(local_yolox):
    shutil.copy(drive_yolox, local_yolox)
    print("📥 YOLOX modeli Drive'dan çekildi.")
elif not os.path.exists(local_yolox):
    raise FileNotFoundError(f"❌ {drive_yolox} konumunda model bulunamadı!")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Hücre 2 Tamamlandı: Veriseti dizinlendi. Çalışma birimi: {device.type.upper()}")

📁 Seçilen Senaryo: MOT17-04-FRCNN
🎞️ Toplam Kare Sayısı: 1050
📊 Ground Truth Konumu: /content/drive/MyDrive/Spikedge_Staj/Tracking/MOT17_Dataset/train/MOT17-04-FRCNN/gt/gt.txt
✅ Hücre 2 Tamamlandı: Veriseti dizinlendi. Çalışma birimi: CUDA


In [26]:
# ==============================================================================
# 🛡️ YENİ HÜCRE 2.5: DEEPRFT ORİJİNAL COMMIT ENTEGRASYONU (DEC 5, 2021)
# ==============================================================================
import os
import sys
import subprocess
import importlib.util
import shutil

print("⚡ [Sistem] DeepRFT orijinal commit (Dec 5, 2021) yapılandırılıyor...", flush=True)

subprocess.run([sys.executable, "-m", "pip", "install", "einops", "timm", "kornia", "ptflops", "natsort", "yacs", "joblib", "addict", "future", "yapf", "lpips", "-q"])

deeprft_dir = "/content/DeepRFT"
if os.path.exists(deeprft_dir):
    shutil.rmtree(deeprft_dir)

print(" 📥 DeepRFT reposu klonlanıyor...", flush=True)
subprocess.run(["git", "clone", "https://github.com/INVOKERer/DeepRFT.git", deeprft_dir])

# 🛠️ Ekran görüntüsündeki orijinal commit'e git!
os.chdir(deeprft_dir)
# Ekran görüntüsünün commit hash'i veya en yakın kararlı sürüm
subprocess.run(["git", "checkout", "a88bf49"])
os.chdir("/content")

if deeprft_dir not in sys.path:
    sys.path.insert(0, deeprft_dir)

try:
    file_path = os.path.join(deeprft_dir, "DeepRFT_MIMO.py")
    if not os.path.exists(file_path):
        raise Exception("DeepRFT_MIMO.py dosyası commit içinde bulunamadı!")

    print(" 🔍 DeepRFT_MIMO.py modülü fiziksel olarak RAM'e alınıyor...", flush=True)
    spec = importlib.util.spec_from_file_location("DeepRFT_MIMO", file_path)
    DeepRFT_MIMO = importlib.util.module_from_spec(spec)
    sys.modules["DeepRFT_MIMO"] = DeepRFT_MIMO
    spec.loader.exec_module(DeepRFT_MIMO)

    import inspect
    DeepRFT_bulundu = None
    for isim, obj in inspect.getmembers(DeepRFT_MIMO, inspect.isclass):
        if 'DeepRFT' in isim:
            DeepRFT_bulundu = obj
            print(f" 🎯 Nokta Atışı Başarılı! Sınıf bulundu: {isim}", flush=True)
            break

    if DeepRFT_bulundu is None:
        raise Exception("DeepRFT sınıfı bulunamadı!")

    import __main__
    __main__.DeepRFT = DeepRFT_bulundu
    print(" 🟢 [BAŞARILI] Orijinal SOTA model Python çekirdeğine MÜHÜRLENDİ!", flush=True)

except Exception as e:
    print(f" ❌ [Kritik Sistem Hatası] {e}", flush=True)

⚡ [Sistem] DeepRFT orijinal commit (Dec 5, 2021) yapılandırılıyor...
 📥 DeepRFT reposu klonlanıyor...
 🔍 DeepRFT_MIMO.py modülü fiziksel olarak RAM'e alınıyor...
 🎯 Nokta Atışı Başarılı! Sınıf bulundu: DeepRFT
 🟢 [BAŞARILI] Orijinal SOTA model Python çekirdeğine MÜHÜRLENDİ!


In [28]:
# ==============================================================================
# ✈️ PRE-FLIGHT CHECK (Senkronizasyon Korumalı)
# ==============================================================================
import torch
import os
import time

weights_path = "/content/drive/MyDrive/Spikedge_Staj/Deblurring/model_GoPro.pth"

print("🔍 [Pre-Flight Check] Drive senkronizasyonu bekleniyor...", flush=True)

# Drive senkronizasyonu için kısa bir bekleme ve kontrol döngüsü
for i in range(10):
    if os.path.exists(weights_path):
        print(f" ✅ Dosya algılandı! Boyut: {os.path.getsize(weights_path) / (1024*1024):.2f} MB", flush=True)
        break
    time.sleep(2)

try:
    import __main__
    if not hasattr(__main__, 'DeepRFT'):
        raise Exception("DeepRFT modeli RAM'de bulunamadı! Lütfen önce Hücre 2.5'i çalıştırın.")

    test_model = __main__.DeepRFT()

    if not os.path.exists(weights_path):
        raise Exception(f"Ağırlık dosyası hâl่ะ senkronize olmadı: {weights_path}")

    checkpoint = torch.load(weights_path, map_location="cpu", weights_only=False)
    raw_state_dict = checkpoint.get("state_dict", checkpoint.get("model", checkpoint))

    clean_state_dict = {k.replace('module.', ''): v for k, v in raw_state_dict.items()}

    test_model.load_state_dict(clean_state_dict, strict=True)

    print("\n" + "🟢"*30, flush=True)
    print(" 🎉 MÜKEMMEL! Altın madeni ağırlık dosyası model mimarisiyle %100 UYUMLU!", flush=True)
    print("🟢"*30, flush=True)

except Exception as e:
    print(f"\n❌ [HATA]:\n{e}", flush=True)

🔍 [Pre-Flight Check] Drive senkronizasyonu bekleniyor...
 ✅ Dosya algılandı! Boyut: 41.51 MB

🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢
 🎉 MÜKEMMEL! Altın madeni ağırlık dosyası model mimarisiyle %100 UYUMLU!
🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢🟢


In [ ]:
# ==============================================================================
# 🚀 HÜCRE 3: ULTIMATE BATCH PIPELINE (SOTA DEEPRFT + MODULE ÖN EK TEMİZLİĞİ)
# ==============================================================================
import os
import time
import cv2
import glob
import numpy as np
import torch
import gc
import sys
import subprocess
import shutil
import configparser

print("🛡️ [Ultimate Enterprise Pipeline] Otonom Mimari, Renk Kalkanı ve Telemetri Mühürleme Aktif...", flush=True)

# 🛠️ FUSE Önbelleğini Ez ve Sıfırdan Başla (Ghosting Kalkanı)
FORCE_CLEAN_RUN = True

subprocess.run([sys.executable, "-m", "pip", "install", "thop", "loguru", "lap", "cython_bbox", "faiss-gpu", "filterpy", "scipy", "-q"])

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
cv2.setNumThreads(1)

def aggressive_ram_purge():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def get_sequence_fps(seq_base_path):
    ini_path = os.path.join(seq_base_path, "seqinfo.ini")
    if os.path.exists(ini_path):
        config = configparser.ConfigParser()
        config.read(ini_path)
        try: return float(config['Sequence']['frameRate'])
        except Exception: pass
    return 30.0

def check_video_health(path):
    if not os.path.exists(path) or os.path.getsize(path) < 10000: return False
    cap = cv2.VideoCapture(path)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return frames > 10

def safe_drive_mirror(local_path, drive_path, stage_name):
    print(f" 🎬 [{stage_name}] H.264 (YUV420P) formatında renk korumalı mühürleniyor...", flush=True)
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    cmd = f"ffmpeg -y -i '{local_path}' -c:v libx264 -pix_fmt yuv420p -preset fast -crf 17 '{drive_path}'"
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if res.returncode != 0 or not os.path.exists(drive_path) or not check_video_health(drive_path):
        shutil.copy(local_path, drive_path)

    time.sleep(1)
    if check_video_health(drive_path):
        size_mb = os.path.getsize(drive_path) / (1024 * 1024)
        print(f" ✅ [{stage_name}] Mühürleme Başarılı! ({size_mb:.2f} MB)", flush=True)
    else:
        print(f" ❌ [{stage_name}] HATA: Drive dizinine sağlıklı yazılamadı!", flush=True)

# ==============================================================================
# 🎛️ MODÜL 1: DEBLURRING (Doğrudan RAM'den Okunan SOTA Mimari - Düzeltilmiş)
# ==============================================================================
def load_deblur_model(weights_path: str, device: str = "cuda"):
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    PROJECT_ROOT = "/content/drive/MyDrive/Spikedge_Staj"
    if not os.path.exists(weights_path) and not os.path.isabs(weights_path):
        weights_path = os.path.join(PROJECT_ROOT, "Deblurring", weights_path)

    try:
        import __main__
        if not hasattr(__main__, 'DeepRFT'):
            raise Exception("Hücre 2.5 çalıştırılmamış! Radar modülü eksik.")

        # Otonom radardan gelen SOTA modelini başlat
        model = __main__.DeepRFT()

        if os.path.exists(weights_path):
            checkpoint = torch.load(weights_path, map_location=device, weights_only=False)
            state_dict = checkpoint.get("state_dict", checkpoint.get("model", checkpoint))

            # 🛠️ Çoklu GPU 'module.' ön ek temizliği (Kritik Düzeltme Eklendi)
            clean_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

            model.load_state_dict(clean_state_dict, strict=True)
            print(" 🟢 [DeepRFT] Orijinal SOTA yapay zeka ağırlıkları başarıyla yüklendi!", flush=True)
        else:
            print(f" ⚠️ [Uyarı] Ağırlık dosyası bulunamadı: {weights_path}", flush=True)

        model.to(device)
        model.eval()
        return model
    except Exception as e:
        print(f" ⚠️ [Kritik] {e}. 'Safe-Sharpen' filtresi aktif edildi.", flush=True)
        return None

def run_deblurring(frames: list, model, device: str = "cuda") -> list:
    frames_arr = np.array(frames)
    deblurred = []

    if model is None:
        for img in frames_arr:
            gaussian = cv2.GaussianBlur(img, (0, 0), 2.0)
            sharpened = cv2.addWeighted(img, 1.5, gaussian, -0.5, 0)
            deblurred.append(np.clip(sharpened, 0, 255).astype(np.uint8))
        return deblurred

    device = torch.device(device if torch.cuda.is_available() else "cpu")
    for img in frames_arr:
        inp = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0) / 255.0
        inp = inp.to(device)
        with torch.no_grad():
            out = model(inp)
            if isinstance(out, (list, tuple)): out = out[0]
        out = torch.clamp(out, 0.0, 1.0)
        out_np = (out.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255.0).astype(np.uint8)
        deblurred.append(out_np)
    return deblurred

# ==============================================================================
# 🎛️ MODÜL 2: LIGHTSTAB STABILIZASYON
# ==============================================================================
def movingAverage(curve, radius):
    window_size = 2 * radius + 1
    f = np.ones(window_size) / window_size
    curve_pad = np.pad(curve, (radius, radius), mode='edge')
    return np.convolve(curve_pad, f, mode='same')[radius:-radius]

def smooth(trajectory, smoothing_radius):
    smoothed_trajectory = np.copy(trajectory)
    for i in range(3): smoothed_trajectory[:, i] = movingAverage(trajectory[:, i], radius=smoothing_radius)
    return smoothed_trajectory

def run_lightstab(input_path, output_path, seq_fps, smoothing_radius=30):
    cap = cv2.VideoCapture(input_path)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), seq_fps, (w, h))
    _, prev = cap.read()
    if prev is None: cap.release(); out.release(); return
    prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    transforms = np.zeros((max(1, n_frames-1), 3), np.float32)

    for i in range(min(n_frames-2, len(transforms))):
        prev_pts = cv2.goodFeaturesToTrack(prev_gray, maxCorners=200, qualityLevel=0.01, minDistance=30, blockSize=3)
        success, curr = cap.read()
        if not success: break
        curr_gray = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY)
        if prev_pts is not None:
            curr_pts, status, err = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_pts, None)
            idx = np.where(status==1)[0]
            if len(idx) > 10:
                prev_pts, curr_pts = prev_pts[idx], curr_pts[idx]
                m, _ = cv2.estimateAffinePartial2D(prev_pts, curr_pts)
                dx, dy, da = (m[0,2], m[1,2], np.arctan2(m[1,0], m[0,0])) if m is not None else (0, 0, 0)
            else: dx, dy, da = 0, 0, 0
        else: dx, dy, da = 0, 0, 0
        transforms[i] = [dx, dy, da]
        prev_gray = curr_gray

    trajectory = np.cumsum(transforms, axis=0)
    transforms_smooth = transforms + (smooth(trajectory, smoothing_radius) - trajectory)

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    for i in range(n_frames-1):
        success, frame = cap.read()
        if not success: break
        dx, dy, da = transforms_smooth[min(i, len(transforms_smooth)-1)]
        m = np.zeros((2,3), np.float32)
        m[0,0], m[0,1], m[0,2] = np.cos(da), -np.sin(da), dx
        m[1,0], m[1,1], m[1,2] = np.sin(da), np.cos(da), dy
        out.write(cv2.warpAffine(frame, m, (w,h)))
    cap.release(); out.release()

# ==============================================================================
# 🔄 DİNAMİK TOPLU İŞLEM DÖNGÜSÜ
# ==============================================================================
PROJECT_ROOT = "/content/drive/MyDrive/Spikedge_Staj"
DRIVE_INTERMEDIATE = os.path.join(PROJECT_ROOT, "Tracking/intermediate")
DRIVE_OUTPUT = os.path.join(PROJECT_ROOT, "Tracking/output_tracks")
DRIVE_TELEMETRY = os.path.join(PROJECT_ROOT, "Tracking/telemetry")
DRIVE_INPUT_VIDEOS = os.path.join(PROJECT_ROOT, "Tracking/input_videos")
LOCAL_DIR = "/content/local_processing"

for d in [LOCAL_DIR, DRIVE_INTERMEDIATE, DRIVE_OUTPUT, DRIVE_TELEMETRY, DRIVE_INPUT_VIDEOS]:
    os.makedirs(d, exist_ok=True)

sequences = ["MOT17-02-FRCNN", "MOT17-04-FRCNN", "MOT17-05-FRCNN", "MOT17-09-FRCNN", "MOT17-10-FRCNN", "MOT17-11-FRCNN", "MOT17-13-FRCNN"]
hybris_dir = "/content/HybridSORT"

if not os.path.exists(hybris_dir):
    subprocess.run(["git", "clone", "https://github.com/ymzis69/HybridSORT.git", hybris_dir])
os.chdir(hybris_dir)

# Yamalar
for py_f in glob.glob("**/*.py", recursive=True):
    try:
        with open(py_f, 'r', encoding='utf-8') as f: content = f.read()
        modified = False
        if "from collections import " in content:
            for old, new in {"from collections import Mapping, OrderedDict": "from collections.abc import Mapping\nfrom collections import OrderedDict", "from collections import Mapping": "from collections.abc import Mapping", "from collections import Iterable": "from collections.abc import Iterable"}.items():
                if old in content: content = content.replace(old, new); modified = True
        if "from torch._six import" in content:
            content = content.replace("from torch._six import string_classes", "string_classes = (str, bytes)").replace("from torch._six import int_classes", "int_classes = int"); modified = True
        if "trk[:] = [pos[0][0]" in content:
            content = content.replace("trk[:] = [pos[0][0], pos[0][1], pos[0][2], pos[0][3], kalman_score, simple_score[0]]", "p_flat = np.atleast_1d(pos[0]).flatten(); s_flat = np.atleast_1d(simple_score).flatten(); trk[:] = [float(p_flat[0]), float(p_flat[1]), float(p_flat[2]), float(p_flat[3]), float(kalman_score), float(s_flat[0])]"); modified = True
        if "torch.load(" in content and "weights_only" not in content:
            content = content.replace('torch.load(ckpt_file, map_location="cpu")', 'torch.load(ckpt_file, map_location="cpu", weights_only=False)').replace("torch.load(ckpt_file, map_location='cpu')", "torch.load(ckpt_file, map_location='cpu', weights_only=False)"); modified = True
        if modified:
            with open(py_f, 'w', encoding='utf-8') as f: f.write(content)
    except Exception: pass

local_ckpt = os.path.join(hybris_dir, "weights/yolox_x.pth")
if not os.path.exists(local_ckpt):
    os.makedirs(os.path.dirname(local_ckpt), exist_ok=True)
    os.system(f"curl -L -# -o {local_ckpt} https://github.com/ifzhang/ByteTrack/releases/download/v0.1_supp/yolox_x.pth")

exp_file = "exps/example/mot/yolox_x_mix_det_hybrid_sort.py"
if not os.path.exists(exp_file): exp_file = "exps/example/mot/yolox_x_mix_det.py"

for seq_name in sequences:
    print(f"\n" + "═"*70, flush=True)
    print(f"🎯 İŞLENEN SEKANS: {seq_name}", flush=True)
    print(f"═"*70, flush=True)

    found_paths = glob.glob(f"/content/drive/MyDrive/**/{seq_name}/img1", recursive=True)
    if not found_paths:
        found_paths = glob.glob(f"/content/drive/MyDrive/**/{seq_name}", recursive=True)
        if found_paths and not os.path.exists(os.path.join(found_paths[0], "img1")): seq_base_path = found_paths[0]
        else: print(f" ⚠️ Uyarı: {seq_name} bulunamadı!", flush=True); continue
    else: seq_base_path = os.path.dirname(found_paths[0])

    seq_fps = get_sequence_fps(seq_base_path)
    print(f" ⚙️ Dinamik FPS Algılandı: {seq_fps}", flush=True)

    image_files = sorted(glob.glob(os.path.join(seq_base_path, "img1", "*.jpg")))
    if not image_files: continue

    temp_source = os.path.join(LOCAL_DIR, f"{seq_name}_source.mp4")
    D_INPUT = os.path.join(DRIVE_INPUT_VIDEOS, f"{seq_name}_raw_input.mp4")

    if not FORCE_CLEAN_RUN and check_video_health(D_INPUT):
        if not os.path.exists(temp_source): shutil.copy(D_INPUT, temp_source)
        print(" ⏭️ [Input Video] Drive'dan yüklendi.", flush=True)
    else:
        print(" 🎞️ [Input Video] Birleştiriliyor...", flush=True)
        sample_img = cv2.imread(image_files[0])
        writer = cv2.VideoWriter(temp_source, cv2.VideoWriter_fourcc(*'mp4v'), seq_fps, (sample_img.shape[1], sample_img.shape[0]))
        for img_path in image_files: writer.write(cv2.imread(img_path))
        writer.release()
        safe_drive_mirror(temp_source, D_INPUT, f"Raw Input ({seq_name})")

    L_STAGE1 = os.path.join(LOCAL_DIR, f"stage1_deblurred_{seq_name}.mp4")
    L_STAGE2 = os.path.join(LOCAL_DIR, f"stage2_stabilized_{seq_name}.mp4")
    D_STAGE1 = os.path.join(DRIVE_INTERMEDIATE, f"stage1_deblurred_{seq_name}.mp4")
    D_STAGE2 = os.path.join(DRIVE_INTERMEDIATE, f"stage2_stabilized_{seq_name}.mp4")
    D_STAGE3 = os.path.join(DRIVE_OUTPUT, f"final_tracked_{seq_name}.mp4")

    # STAGE 1: DEBLURRING
    if not FORCE_CLEAN_RUN and check_video_health(D_STAGE1):
        if not os.path.exists(L_STAGE1): shutil.copy(D_STAGE1, L_STAGE1)
        print(" ⏭️ [Stage 1] Atlanıyor.", flush=True)
    else:
        print(f" 🚀 [Stage 1] DeepRFT SOTA Başlatılıyor...", flush=True)
        m1 = load_deblur_model("model_GoPro.pth", device="cuda")
        cap = cv2.VideoCapture(temp_source)
        writer_s1 = cv2.VideoWriter(L_STAGE1, cv2.VideoWriter_fourcc(*'mp4v'), seq_fps, (sample_img.shape[1], sample_img.shape[0]))
        chunk = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            chunk.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if len(chunk) >= 100:
                for f in run_deblurring(chunk, m1, device="cuda"): writer_s1.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
                chunk.clear(); aggressive_ram_purge()
        if chunk:
            for f in run_deblurring(chunk, m1, device="cuda"): writer_s1.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
        cap.release(); writer_s1.release()
        try: del m1
        except: pass
        aggressive_ram_purge()
        safe_drive_mirror(L_STAGE1, D_STAGE1, f"Stage 1 ({seq_name})")

    # STAGE 2: STABILIZASYON
    if not FORCE_CLEAN_RUN and check_video_health(D_STAGE2):
        if not os.path.exists(L_STAGE2): shutil.copy(D_STAGE2, L_STAGE2)
        print(" ⏭️ [Stage 2] Atlanıyor.", flush=True)
    else:
        print(f" 🚀 [Stage 2] LightStab Çalıştırılıyor...", flush=True)
        run_lightstab(L_STAGE1, L_STAGE2, seq_fps, smoothing_radius=30)
        safe_drive_mirror(L_STAGE2, D_STAGE2, f"Stage 2 ({seq_name})")
    aggressive_ram_purge()

    # STAGE 3: TRACKING
    if not FORCE_CLEAN_RUN and check_video_health(D_STAGE3):
        print(" ⏭️ [Stage 3] Final Video hazır, atlanıyor.", flush=True)
    else:
        print(f" 🚀 [Stage 3] Hybrid-SORT Takip Motoru...", flush=True)
        sota_out_dir = f"/content/sota_run_out_{seq_name}"
        os.makedirs(sota_out_dir, exist_ok=True)

        cmd = [sys.executable, f"{hybris_dir}/tools/demo_track.py", "--demo_type", "video", "-f", exp_file, "-c", local_ckpt, "--path", L_STAGE2, "--output_dir", sota_out_dir, "--device", "gpu", "--fp16", "--fuse", "--save_result"]
        env = os.environ.copy(); env["PYTHONPATH"] = hybris_dir
        process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            if "Processing frame" in line or "save results to" in line: sys.stdout.write(line); sys.stdout.flush()
        process.wait()

        if process.returncode != 0: print(f" ❌ {seq_name} Hybrid-SORT hatası!", flush=True); continue

        # TELEMETRİ
        txt_files = glob.glob(os.path.join(sota_out_dir, "**/track_vis/*.txt"), recursive=True) + glob.glob("/content/HybridSORT/YOLOX_outputs/**/track_vis/*.txt", recursive=True)
        if txt_files:
            latest_txt = max(txt_files, key=os.path.getmtime)
            shutil.copy(latest_txt, os.path.join(DRIVE_TELEMETRY, f"{seq_name}.txt"))
            print(f" 💾 [Telemetri] Drive'a kilitlendi: {seq_name}.txt", flush=True)
        else: continue

        print(f" 🎨 [Stage 3.5] Render Stüdyosu...", flush=True)
        tracking_data = {}
        with open(latest_txt, 'r') as f:
            for line in f:
                parts = line.strip().replace(',', ' ').split()
                if len(parts) < 6: continue
                frame_id = int(float(parts[0]))
                if frame_id == 0: frame_id += 1
                track_id = int(float(parts[1]))
                if frame_id not in tracking_data: tracking_data[frame_id] = []
                tracking_data[frame_id].append((track_id, float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])))

        cap = cv2.VideoCapture(L_STAGE2)
        local_rendered = os.path.join(LOCAL_DIR, f"rendered_{seq_name}.mp4")
        writer = cv2.VideoWriter(local_rendered, cv2.VideoWriter_fourcc(*'mp4v'), seq_fps, (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))

        np.random.seed(42); colors = {}
        f_idx = 1
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            if f_idx in tracking_data:
                for tid, l, t, wd, ht in tracking_data[f_idx]:
                    x1, y1, x2, y2 = int(l), int(t), int(l + wd), int(t + ht)
                    if tid not in colors: colors[tid] = (int(np.random.randint(50, 255)), int(np.random.randint(50, 255)), int(np.random.randint(50, 255)))
                    cv2.rectangle(frame, (x1, y1), (x2, y2), colors[tid], 2)
                    label = f"ID: {tid}"
                    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                    cv2.rectangle(frame, (x1, y1 - 18), (x1 + tw + 4, y1), colors[tid], -1)
                    cv2.putText(frame, label, (x1 + 2, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            writer.write(frame); f_idx += 1
        cap.release(); writer.release()
        safe_drive_mirror(local_rendered, D_STAGE3, f"Final Tracked ({seq_name})")
        aggressive_ram_purge()

os.chdir("/content")
print("\n" + "🏆"*35, flush=True)
print(" 🎉 TÜM MOT17 EĞİTİM SETİ (GERÇEK SOTA, DİNAMİK FPS VE RENK KORUMASIYLA) MÜHÜRLENDİ! 🎉", flush=True)
print("🏆"*35, flush=True)

🛡️ [Ultimate Enterprise Pipeline] Otonom Mimari, Renk Kalkanı ve Telemetri Mühürleme Aktif...

══════════════════════════════════════════════════════════════════════
🎯 İŞLENEN SEKANS: MOT17-02-FRCNN
══════════════════════════════════════════════════════════════════════
 ⚙️ Dinamik FPS Algılandı: 30.0
 🎞️ [Input Video] Birleştiriliyor...
 🎬 [Raw Input (MOT17-02-FRCNN)] H.264 (YUV420P) formatında renk korumalı mühürleniyor...
 ✅ [Raw Input (MOT17-02-FRCNN)] Mühürleme Başarılı! (35.44 MB)
 🚀 [Stage 1] DeepRFT SOTA Başlatılıyor...
 🟢 [DeepRFT] Orijinal SOTA yapay zeka ağırlıkları başarıyla yüklendi!
 🎬 [Stage 1 (MOT17-02-FRCNN)] H.264 (YUV420P) formatında renk korumalı mühürleniyor...
 ✅ [Stage 1 (MOT17-02-FRCNN)] Mühürleme Başarılı! (36.39 MB)
 🚀 [Stage 2] LightStab Çalıştırılıyor...
 🎬 [Stage 2 (MOT17-02-FRCNN)] H.264 (YUV420P) formatında renk korumalı mühürleniyor...
 ✅ [Stage 2 (MOT17-02-FRCNN)] Mühürleme Başarılı! (36.12 MB)
 🚀 [Stage 3] Hybrid-SORT Takip Motoru...
2026-07-31 16:22:25

In [ ]:
# ==============================================================================
# 📈 BATCH EVALUATION: AKILLI ZAMAN ÇİZELGESİ (CHRONO-MAPPING) VE SOTA ÖZETİ
# ==============================================================================
import os
import sys
import subprocess
import glob
import shutil

print("📊 [Batch Evaluation Engine] Zaman Çizelgeli Telemetri Eşleştirici Başlatılıyor...\n")

trackeval_path = "/content/TrackEval"
if not os.path.exists(trackeval_path):
    print(" 📥 TrackEval klonlanıyor...")
    subprocess.run(["git", "clone", "https://github.com/JonathonLuiten/TrackEval.git", trackeval_path])

sequences = [
    "MOT17-02-FRCNN",
    "MOT17-04-FRCNN",
    "MOT17-05-FRCNN",
    "MOT17-09-FRCNN",
    "MOT17-10-FRCNN",
    "MOT17-11-FRCNN",
    "MOT17-13-FRCNN"
]

tracker_name = "HybridSORT_Official"
data_dir = os.path.join(trackeval_path, "data/trackers/mot_challenge/MOT17-train", tracker_name, "data")
os.makedirs(data_dir, exist_ok=True)

print(" 🔍 YOLOX Çıktı Klasöründeki Zaman Damgalı Telemetriler Taranıyor...")

# YOLOX'un kendi klasöründeki tüm txt dosyalarını bul ve değiştirilme tarihine göre sırala
all_txt_files = glob.glob("/content/HybridSORT/YOLOX_outputs/**/track_vis/*.txt", recursive=True)
all_txt_files.sort(key=os.path.getmtime)

found_count = 0
if len(all_txt_files) >= len(sequences):
    # Chrono-Mapping: İşlem sırasına göre son 7 dosyayı alıyoruz
    target_txt_files = all_txt_files[-len(sequences):]

    for seq, txt_file in zip(sequences, target_txt_files):
        dest_txt = os.path.join(data_dir, f"{seq}.txt")
        shutil.copy(txt_file, dest_txt)
        print(f"   ✔️ {os.path.basename(txt_file)} --> {seq}.txt olarak eşleştirildi.")
        found_count += 1
else:
    print(f" ❌ Kritik Hata: Yeterli telemetri dosyası bulunamadı. Beklenen: {len(sequences)}, Bulunan: {len(all_txt_files)}")

print(f"\n 🎯 Toplam {found_count}/{len(sequences)} sekans TrackEval'e yüklendi.\n")

if found_count == len(sequences):
    print(" 🚀 TrackEval Kümülatif Motoru Çalıştırılıyor (Lütfen bekleyiniz)...\n")
    eval_cmd = [
        sys.executable, os.path.join(trackeval_path, "scripts/run_mot_challenge.py"),
        "--BENCHMARK", "MOT17",
        "--SPLIT_TO_EVAL", "train",
        "--TRACKERS_TO_EVAL", tracker_name,
        "--METRICS", "HOTA", "CLEAR", "Identity",
        "--USE_PARALLEL", "False",
        "--PRINT_RESULTS", "False"
    ]

    env = os.environ.copy()
    res = subprocess.run(eval_cmd, cwd=trackeval_path, capture_output=True, text=True, env=env)

    # --- METRİK PARSER ---
    hota, mota, idf1, idsw, motp = "N/A", "N/A", "N/A", "N/A", "N/A"
    current_metric = None

    for line in res.stdout.split('\n'):
        if "HOTA: " in line: current_metric = "HOTA"
        elif "CLEAR: " in line: current_metric = "CLEAR"
        elif "Identity: " in line: current_metric = "Identity"

        if line.startswith("COMBINED"):
            parts = line.split()
            if current_metric == "HOTA" and len(parts) > 1:
                hota = parts[1]
            elif current_metric == "CLEAR" and len(parts) > 13:
                mota = parts[1]
                motp = parts[2]
                idsw = parts[13]
            elif current_metric == "Identity" and len(parts) > 1:
                idf1 = parts[1]

    # --- EKRAN ÇIKTISI (DASHBOARD) ---
    print("="*85)
    print(" 🏆 MOT17 TRAIN GLOBAL SOTA DEĞERLENDİRME RAPORU (7 SEKANS BİRLEŞİK)")
    print("="*85)
    print(" 🎛️ BORU HATTI (PIPELINE) BİLEŞENLERİ & PERFORMANS ETKİLERİ:")
    print("   • Stage 1 (Ön İşleme)    : DeepRFT Neural Deblurring")
    print("                              -> Hızlı hareketlerdeki motion blur'u gidererek FN'i düşürür.")
    print("   • Stage 2 (Stabilizasyon): LightStab Optical Flow")
    print("                              -> Kamera sarsıntılarını izole ederek Bounding Box titremesini önler.")
    print("   • Stage 3 (Takip Motoru) : Hybrid-SORT (YOLOX-X + Kalman Filtresi)")
    print("                              -> ID sürekliliğini sağlar ve IDSW skorunu minimize eder.")
    print("-" * 85)
    print(" 📊 KÜMÜLATİF (COMBINED) AKADEMİK METRİKLER:")
    print(f"   ➤ HOTA (Genel Başarı)     : {hota} %")
    print(f"   ➤ MOTA (Takip Doğruluğu)  : {mota} %")
    print(f"   ➤ IDF1 (Kimlik Koruma)    : {idf1} %")
    print(f"   ➤ MOTP (Kutu Hassasiyeti) : {motp} %")
    print(f"   ➤ IDSW (Kimlik Değişimi)  : {idsw} (Tüm veri seti toplam ID kopma sayısı)")
    print("="*85)
    print(" 💡 Mühendislik Notu: Bu değerler aritmetik ortalama değil, 7 sekansın tamamındaki TP/FP/FN")
    print("    sayılarının tek bir global havuzda toplanmasıyla hesaplanan GERÇEK KÜMÜLATİF skordur.")
    print("="*85)

📊 [Batch Evaluation Engine] Zaman Çizelgeli Telemetri Eşleştirici Başlatılıyor...

 📥 TrackEval klonlanıyor...
 🔍 YOLOX Çıktı Klasöründeki Zaman Damgalı Telemetriler Taranıyor...
 ❌ Kritik Hata: Yeterli telemetri dosyası bulunamadı. Beklenen: 7, Bulunan: 0

 🎯 Toplam 0/7 sekans TrackEval'e yüklendi.

